<a href="https://colab.research.google.com/github/Datahuntl/Estudo-Comparativo-de-Detec-o-de-DeepFakes/blob/main/CNN_LSTM_Temporal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install timm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import os
import cv2
import numpy as np
from PIL import Image
from google.colab import drive
import timm

In [ ]:
drive.mount('/content/drive')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando na instância: {device}")

In [ ]:
PATH_REAL_VIDEOS = "/content/drive/MyDrive/IC DeepFakes/DATASET/ORIGINAL/REAL"
PATH_FAKE_VIDEOS = "/content/drive/MyDrive/IC DeepFakes/DATASET/ORIGINAL/FAKE"

In [ ]:
class TemporalDeepfakeDataset(Dataset):
    def __init__(self, real_dir, fake_dir, sequence_length=10, transform=None):
        self.real_videos = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.lower().endswith('.mp4')]
        self.fake_videos = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.lower().endswith('.mp4')]

        self.all_videos = self.real_videos + self.fake_videos
        self.labels = [0] * len(self.real_videos) + [1] * len(self.fake_videos)

        self.sequence_length = sequence_length
        self.transform = transform

        # Carregar classificador Haar Cascade básico para recortar a face em tempo de execução
        self.face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    def __len__(self):
        return len(self.all_videos)

    def __getitem__(self, idx):
        video_path = self.all_videos[idx]
        label = self.labels[idx]

        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # Seleciona frames igualmente espaçados para cobrir o vídeo todo temporalmente
        if total_frames > self.sequence_length:
            frame_indices = np.linspace(0, total_frames - 1, self.sequence_length, dtype=int)
        else:
            frame_indices = range(max(1, total_frames))

        frames_seq = []
        current_frame_idx = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if current_frame_idx in frame_indices:
                # Converter para escala de cinza para detectar a face rapidamente
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                faces = self.face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(60, 60))

                # Se detectar o rosto, recorta. Se não, usa o frame central inteiro redimensionado
                if len(faces) > 0:
                    (x, y, w, h) = faces[0]
                    face_crop = frame[y:y+h, x:x+w]
                else:
                    face_crop = frame

                # Converter para PIL e aplicar transformações espaciais
                img_pil = Image.fromarray(cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB))
                if self.transform:
                    img_tensor = self.transform(img_pil)
                frames_seq.append(img_tensor)

            current_frame_idx += 1
            if len(frames_seq) == self.sequence_length:
                break

        cap.release()

        # Preenchimento (padding) caso o vídeo seja extremamente curto e não atinja o tamanho da sequência
        while len(frames_seq) < self.sequence_length:
            frames_seq.append(torch.zeros((3, 224, 224)))

        # Empilha os frames gerando um tensor final no formato: (Sequencia, Canais, Altura, Largura)
        frames_tensor = torch.stack(frames_seq)
        return frames_tensor, label

# Transformação padrão espacial (Reduzido para 224x224 devido ao peso do modelo temporal)
transform_temporal = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = TemporalDeepfakeDataset(real_dir=PATH_REAL_VIDEOS, fake_dir=PATH_FAKE_V_VIDEOS, sequence_length=10, transform=transform_temporal)

# Divisão de Treino e Validação
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# Batch size pequeno (4 ou 8) é obrigatório aqui! Tensores temporais 5D consomem muita VRAM
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print(f"✅ Dataset Temporal CNN-LSTM pronto para carregar sequências de vídeos!")

In [ ]:
class CNNLSTMModel(nn.Module):
    def __init__(self):
        super(CNNLSTMModel, self).__init__()
        # 1. Extrator de características espaciais (Backbone CNN)
        # Usamos uma ResNet-18 leve pois ela processará múltiplos frames por segundo
        self.cnn = timm.create_model('resnet18', pretrained=True)
        num_features = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity() # Remove a camada linear original da CNN

        # 2. Camada Temporal (LSTM)
        # Processará a sequência de características vetorizadas geradas pela ResNet
        self.lstm = nn.LSTM(input_size=num_features, hidden_size=256, num_layers=1, batch_first=True)

        # 3. Classificador Final Binário
        self.classifier = nn.Sequential(
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x tem o formato: (Batch, Sequencia, Canais, Altura, Largura)
        batch_size, seq_len, c, h, w = x.size()

        # Achata as dimensões de Batch e Sequência para que a CNN processe todos os frames de uma vez
        x = x.view(batch_size * seq_len, c, h, w)
        spatial_features = self.cnn(x) # Saída: (Batch * Sequencia, Num_Features)

        # Restaura o formato original para a entrada da rede temporal LSTM
        spatial_features = spatial_features.view(batch_size, seq_len, -1)

        # Passa pela LSTM
        lstm_out, (hidden, cell) = self.lstm(spatial_features)

        # Pega apenas a saída do ÚLTIMO frame da sequência temporal para classificar o vídeo
        last_frame_output = lstm_out[:, -1, :]

        out = self.classifier(last_frame_output)
        return out

model = CNNLSTMModel().to(device)
print("✅ Rede Neural Híbrida CNN-LSTM inicializada com sucesso!")

In [ ]:
import matplotlib.pyplot as plt
import time
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score

CHECKPOINT_DIR = "/content/drive/MyDrive/IC DeepFakes/CNNLSTM_Outputs"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

checkpoint_path = os.path.join(CHECKPOINT_DIR, "cnnlstm_best_checkpoint.pth")
grafico_loss_path = os.path.join(CHECKPOINT_DIR, "curva_perda_cnnlstm.png")
grafico_acc_path = os.path.join(CHECKPOINT_DIR, "curva_acuracia_cnnlstm.png")

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00005) # Taxa de aprendizado baixa para redes complexas
epochs = 10
start_epoch = 0
best_val_acc = 0.0

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

if os.path.exists(checkpoint_path):
    print(f"🔄 Checkpoint temporal encontrado em: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint['best_val_acc']
    history = checkpoint['history']
    print(f"▶️ Resumindo a partir da Época [{start_epoch+1}/{epochs}]")

print("🚀 Iniciando treinamento temporal complexo (CNN-LSTM)...\n")
for epoch in range(start_epoch, epochs):
    start_time = time.time()
    model.train()
    train_loss = 0.0
    all_train_labels, all_train_preds = [], []
    train_bar = tqdm(train_loader, desc=f"🎬 Época {epoch+1}/{epochs} [Treino LSTM]", leave=False)

    for batch_idx, (inputs, labels) in enumerate(train_bar):
        inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        train_loss += current_loss * inputs.size(0)
        preds_binary = (outputs.cpu().detach().numpy() > 0.5).astype(int)
        all_train_labels.extend(labels.cpu().numpy())
        all_train_preds.extend(preds_binary)

        if batch_idx % 2 == 0:
            batch_acc = accuracy_score(labels.cpu().numpy(), preds_binary)
            train_bar.set_postfix({"Loss": f"{current_loss:.4f}", "Acc": f"{batch_acc*100:.1f}%"})

    epoch_train_loss = train_loss / len(train_loader.dataset)
    epoch_train_acc = accuracy_score(all_train_labels, all_train_preds)

    # Validação
    model.eval()
    val_loss = 0.0
    all_labels, all_preds = [], []
    val_bar = tqdm(val_loader, desc=f"🔍 Época {epoch+1}/{epochs} [Validação LSTM]", leave=False)

    with torch.no_grad():
        for inputs, labels in val_bar:
            inputs, labels_eval = inputs.to(device), labels.to(device).float().unsqueeze(1)
            outputs = model(inputs)
            loss = criterion(outputs, labels_eval)
            val_loss += loss.item() * inputs.size(0)
            all_labels.extend(labels.numpy())
            all_preds.extend(outputs.cpu().numpy())

    epoch_val_loss = val_loss / len(val_loader.dataset)
    all_labels, all_preds = np.array(all_labels), np.array(all_preds)
    binary_preds = (all_preds > 0.5).astype(int)

    epoch_val_acc = accuracy_score(all_labels, binary_preds)
    f1 = f1_score(all_labels, binary_preds, zero_division=0)
    prec = precision_score(all_labels, binary_preds, zero_division=0)
    rec = recall_score(all_labels, binary_preds, zero_division=0)
    try: auc = roc_auc_score(all_labels, all_preds)
    except: auc = 0.5

    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_acc"].append(epoch_val_acc)

    epoch_time = time.time() - start_time
    print(f"📊 [FIM DA ÉPOCA {epoch+1}/{epochs}] Tempo: {epoch_time:.1f}s")
    print(f"   📈 Treino     -> Perda: {epoch_train_loss:.4f} | Acurácia: {epoch_train_acc*100:.2f}%")
    print(f"   📉 Validação  -> Perda: {epoch_val_loss:.4f} | Acurácia: {epoch_val_acc*100:.2f}%")
    print(f"   ✨ Métricas   -> AUC-ROC: {auc:.4f} | F1-Score: {f1:.4f}")

    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        print(f"   💾 ⭐ Novo recorde de acurácia! Salvando pesos no Drive...")
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'best_val_acc': best_val_acc, 'history': history}, checkpoint_path)
    print("-" * 80)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(history["train_loss"])+1), history["train_loss"], label="Treino", color="blue", linestyle="--")
plt.plot(range(1, len(history["val_loss"])+1), history["val_loss"], label="Validação", color="red")
plt.title("Evolução da Perda - CNN-LSTM")
plt.grid(True); plt.legend(); plt.savefig(grafico_loss_path, dpi=300); plt.show()

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(history["train_acc"])+1), history["train_acc"], label="Treino", color="blue", linestyle="--")
plt.plot(range(1, len(history["val_acc"])+1), history["val_acc"], label="Validação", color="green")
plt.title("Evolução da Acurácia - CNN-LSTM")
plt.grid(True); plt.legend(); plt.savefig(grafico_acc_path, dpi=300); plt.show()